# Silver – FACT_MEDICATION_REVIEWS
Goal: Produce a cleaned and standardized medication reviews table for analytics.

Input: capstone.bronze.medication_reviews_raw
Output: capstone.silver.fact_medication_reviews

Key steps:
select → clean/standardize → safe parsing (dates & numbers) → generate IDs → deduplicate → write

In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_tbl = "capstone.bronze.medication_reviews_raw"
silver_tbl = "capstone.silver.fact_medication_reviews"

df = spark.table(bronze_tbl)

# -------------------------
# 1) Select only what we need
# -------------------------
keep = [
    "MedicineName",
    "MedicineFor",
    "ReviewDate",
    "UserName",
    "IntakeTime",
    "Reviews",
    "ReviewLength",
    "Rating",
    "NumberOfLikes",
    "ingestion_timestamp",
    "source_system"
]
df = df.select([c for c in keep if c in df.columns])

# -------------------------
# 2) Basic cleaning
# -------------------------
def clean_str(col):
    return F.when(F.trim(F.col(col)) == "", None).otherwise(F.trim(F.col(col)))

df = (df
    .withColumn("medicine_name", clean_str("MedicineName"))
    .withColumn("medicine_for", clean_str("MedicineFor"))
    .withColumn("user_name", clean_str("UserName"))
    .withColumn("review_text", clean_str("Reviews"))
    .withColumn("intake_time", clean_str("IntakeTime"))
)

# Keep raw numeric fields cleaned (so "" becomes NULL)
df = (df
    .withColumn("rating_raw", clean_str("Rating"))
    .withColumn("review_length_raw", clean_str("ReviewLength"))
    .withColumn("likes_raw", clean_str("NumberOfLikes"))
)

# -------------------------
# 3) Safe date parsing
# -------------------------
df = df.withColumn(
    "review_date",
    F.coalesce(
        F.try_to_date(F.col("ReviewDate"), "d-MMM-yy"),
        F.try_to_date(F.col("ReviewDate"), "dd-MMM-yy"),
        F.try_to_date(F.col("ReviewDate"), "d-MMM-yyyy"),
        F.try_to_date(F.col("ReviewDate"), "dd-MMM-yyyy"),
        F.try_to_date(F.col("ReviewDate"))
    )
)

# -------------------------
# 4) Safe numeric parsing (NO hard casts)
# -------------------------
# 4.1 try_cast directly
df = (df
    .withColumn("rating", F.expr("try_cast(rating_raw as double)"))
    .withColumn("review_length", F.expr("try_cast(review_length_raw as int)"))
    .withColumn("likes_count", F.expr("try_cast(likes_raw as int)"))
)

# 4.2 fallback: extract digits -> try_cast (still safe if no digits)
df = (df
    .withColumn(
        "review_length",
        F.coalesce(
            F.col("review_length"),
            F.expr("try_cast(regexp_extract(review_length_raw, '(\\\\d+)', 1) as int)")
        )
    )
    .withColumn(
        "likes_count",
        F.coalesce(
            F.col("likes_count"),
            F.expr("try_cast(regexp_extract(likes_raw, '(\\\\d+)', 1) as int)")
        )
    )
)

# -------------------------
# 5) Generate stable IDs
# -------------------------
df = df.withColumn("medication_id", F.sha2(F.col("medicine_name"), 256))

review_key = F.concat_ws(
    "|",
    F.col("medicine_name"),
    F.col("user_name"),
    F.col("review_text"),
    F.col("review_date").cast("string")
)
df = df.withColumn("review_id", F.sha2(review_key, 256))

# -------------------------
# 6) Deduplicate (keep latest by ingestion_timestamp)
# -------------------------
w = Window.partitionBy("review_id").orderBy(F.col("ingestion_timestamp").desc_nulls_last())

df = (df
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# -------------------------
# 7) Final columns
# -------------------------
silver_df = df.select(
    "review_id",
    "medication_id",
    "medicine_name",
    "medicine_for",
    "user_name",
    "review_text",
    "review_date",
    "intake_time",
    "rating",
    "review_length",
    "likes_count",
    "ingestion_timestamp",
    "source_system"
)

# -------------------------
# 8) Write to Silver (Delta)
# -------------------------
spark.sql("CREATE SCHEMA IF NOT EXISTS capstone.silver")

(silver_df
 .write
 .mode("overwrite")
 .format("delta")
 .saveAsTable(silver_tbl)
)

display(spark.table(silver_tbl).limit(20))

In [0]:
%sql

-- ===============================
-- 1. Total Rows
-- ===============================
SELECT 
'Total Rows' AS check_type,
COUNT(*) AS result
FROM capstone.silver.fact_medication_reviews

UNION ALL

-- ===============================
-- 2. Missing Medicine Name
-- ===============================
SELECT
'Missing Medicine Name',
COUNT(*)
FROM capstone.silver.fact_medication_reviews
WHERE medicine_name IS NULL

UNION ALL

-- ===============================
-- 3. Missing Review Text
-- ===============================
SELECT
'Missing Review Text',
COUNT(*)
FROM capstone.silver.fact_medication_reviews
WHERE review_text IS NULL

UNION ALL

-- ===============================
-- 4. Missing Review Date
-- ===============================
SELECT
'Missing Review Date',
COUNT(*)
FROM capstone.silver.fact_medication_reviews
WHERE review_date IS NULL

UNION ALL

-- ===============================
-- 5. Missing Rating
-- ===============================
SELECT
'Missing Rating',
COUNT(*)
FROM capstone.silver.fact_medication_reviews
WHERE rating IS NULL

UNION ALL

-- ===============================
-- 6. Invalid Rating (<0 or >10)
-- ===============================
SELECT
'Invalid Rating',
COUNT(*)
FROM capstone.silver.fact_medication_reviews
WHERE rating < 0 OR rating > 10

UNION ALL

-- ===============================
-- 7. Duplicate Reviews
-- ===============================
SELECT
'Duplicate Review IDs',
COUNT(*)
FROM (
SELECT review_id
FROM capstone.silver.fact_medication_reviews
GROUP BY review_id
HAVING COUNT(*) > 1
)

UNION ALL

-- ===============================
-- 8. Null Medication IDs
-- ===============================
SELECT
'Missing Medication ID',
COUNT(*)
FROM capstone.silver.fact_medication_reviews
WHERE medication_id IS NULL

UNION ALL

-- ===============================
-- 9. Missing Ingestion Timestamp
-- ===============================
SELECT
'Missing Ingestion Timestamp',
COUNT(*)
FROM capstone.silver.fact_medication_reviews
WHERE ingestion_timestamp IS NULL;